In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import copy
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

from PIL import Image
import pywt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

PROJECT_DIR = "/content/drive/MyDrive/Multimedia_Authentication"

FACES_DIR = os.path.join(
    PROJECT_DIR,
    "faces"
)

SPLIT_FILE = os.path.join(
    PROJECT_DIR,
    "splits",
    "ffpp_2647_processed_split.csv"
)

SPATIAL_CHECKPOINT = os.path.join(
    PROJECT_DIR,
    "spatial_resnet18_best.pth"
)

FREQUENCY_CHECKPOINT = os.path.join(
    PROJECT_DIR,
    "frequency_resnet18_best.pth"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Project exists:", os.path.exists(PROJECT_DIR))
print("Faces exists:", os.path.exists(FACES_DIR))
print("Split exists:", os.path.exists(SPLIT_FILE))
print("Spatial checkpoint:", os.path.exists(SPATIAL_CHECKPOINT))
print("Frequency checkpoint:", os.path.exists(FREQUENCY_CHECKPOINT))

Device: cuda
Project exists: True
Faces exists: True
Split exists: True
Spatial checkpoint: True
Frequency checkpoint: True


In [3]:
df = pd.read_csv(SPLIT_FILE)

print("Total rows:", len(df))
print(df["split"].value_counts())
print(df["Label"].value_counts())

Total rows: 2647
split
train    1822
test      424
val       401
Name: count, dtype: int64
Label
FAKE    2211
REAL     436
Name: count, dtype: int64


In [4]:
def make_output_name(file_path):
    return (
        file_path
        .replace("/", "__")
        .replace("\\", "__")
        .replace(".mp4", ".npz")
    )

In [5]:
df["face_file"] = df["File Path"].apply(
    make_output_name
)

face_files = {
    f for f in os.listdir(FACES_DIR)
    if f.endswith(".npz")
}

df["face_exists"] = df["face_file"].isin(
    face_files
)

train_df = df[
    (df["split"] == "train") &
    (df["face_exists"])
].copy()

val_df = df[
    (df["split"] == "val") &
    (df["face_exists"])
].copy()

test_df = df[
    (df["split"] == "test") &
    (df["face_exists"])
].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 1770
Validation: 392
Test: 418


In [6]:
def dwt_high_frequency(frame):

    frame = frame.astype(
        np.float32
    ) / 255.0

    LH_channels = []
    HL_channels = []
    HH_channels = []

    for c in range(3):

        channel = frame[:, :, c]

        LL, (LH, HL, HH) = pywt.dwt2(
            channel,
            "haar"
        )

        LH_channels.append(LH)
        HL_channels.append(HL)
        HH_channels.append(HH)

    LH = np.mean(
        np.stack(LH_channels),
        axis=0
    )

    HL = np.mean(
        np.stack(HL_channels),
        axis=0
    )

    HH = np.mean(
        np.stack(HH_channels),
        axis=0
    )

    return np.stack(
        [LH, HL, HH],
        axis=-1
    ).astype(np.float32)

In [7]:
def normalize_frequency_image(freq_img):

    min_val = freq_img.min()
    max_val = freq_img.max()

    freq_img = (
        freq_img - min_val
    ) / (
        max_val - min_val + 1e-8
    )

    freq_img = (
        freq_img * 255
    ).clip(
        0,
        255
    ).astype(
        np.uint8
    )

    return Image.fromarray(freq_img)

In [8]:
class SpatialFrequencyDataset(Dataset):

    def __init__(
        self,
        dataframe,
        faces_dir,
        rgb_transform=None,
        frequency_transform=None
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.faces_dir = faces_dir

        self.rgb_transform = rgb_transform
        self.frequency_transform = frequency_transform

        self.label_map = {
            "REAL": 0,
            "FAKE": 1
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        npz_path = os.path.join(
            self.faces_dir,
            row["face_file"]
        )

        data = np.load(npz_path)

        frames = data["frames"]

        rgb_frames = []
        frequency_frames = []

        for frame in frames:

            # -------------------------
            # Spatial RGB representation
            # -------------------------

            rgb_image = Image.fromarray(
                frame
            )

            if self.rgb_transform:
                rgb_image = self.rgb_transform(
                    rgb_image
                )

            rgb_frames.append(
                rgb_image
            )

            # -------------------------
            # Frequency representation
            # -------------------------

            freq = dwt_high_frequency(
                frame
            )

            freq_image = normalize_frequency_image(
                freq
            )

            freq_image = freq_image.resize(
                (224, 224)
            )

            if self.frequency_transform:
                freq_image = self.frequency_transform(
                    freq_image
                )

            frequency_frames.append(
                freq_image
            )

        rgb_frames = torch.stack(
            rgb_frames
        )

        frequency_frames = torch.stack(
            frequency_frames
        )

        label = self.label_map[
            row["Label"]
        ]

        return (
            rgb_frames,
            frequency_frames,
            torch.tensor(
                label,
                dtype=torch.long
            )
        )

In [9]:
rgb_transform = transforms.Compose([

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

In [10]:
frequency_transform = transforms.Compose([

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

In [11]:
fusion_train_dataset = SpatialFrequencyDataset(
    train_df,
    FACES_DIR,
    rgb_transform=rgb_transform,
    frequency_transform=frequency_transform
)

fusion_val_dataset = SpatialFrequencyDataset(
    val_df,
    FACES_DIR,
    rgb_transform=rgb_transform,
    frequency_transform=frequency_transform
)

fusion_test_dataset = SpatialFrequencyDataset(
    test_df,
    FACES_DIR,
    rgb_transform=rgb_transform,
    frequency_transform=frequency_transform
)

print(
    "Train:",
    len(fusion_train_dataset)
)

print(
    "Validation:",
    len(fusion_val_dataset)
)

print(
    "Test:",
    len(fusion_test_dataset)
)

Train: 1770
Validation: 392
Test: 418


In [12]:
rgb_sample, freq_sample, label = (
    fusion_train_dataset[0]
)

print(
    "RGB shape:",
    rgb_sample.shape
)

print(
    "Frequency shape:",
    freq_sample.shape
)

print(
    "Label:",
    label
)

RGB shape: torch.Size([8, 3, 224, 224])
Frequency shape: torch.Size([8, 3, 224, 224])
Label: tensor(1)


In [13]:
fusion_train_loader = DataLoader(
    fusion_train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

fusion_val_loader = DataLoader(
    fusion_val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

fusion_test_loader = DataLoader(
    fusion_test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [14]:
rgb_batch, freq_batch, labels = next(
    iter(fusion_train_loader)
)

print(
    "RGB batch:",
    rgb_batch.shape
)

print(
    "Frequency batch:",
    freq_batch.shape
)

print(
    "Labels:",
    labels.shape
)

print(
    "Labels:",
    labels
)

RGB batch: torch.Size([4, 8, 3, 224, 224])
Frequency batch: torch.Size([4, 8, 3, 224, 224])
Labels: torch.Size([4])
Labels: tensor([1, 1, 1, 1])


In [15]:
class ResNet18FeatureExtractor(nn.Module):

    def __init__(self):
        super().__init__()

        backbone = resnet18(
            weights=ResNet18_Weights.DEFAULT
        )

        # Remove the final classification layer
        self.features = nn.Sequential(
            *list(backbone.children())[:-1]
        )

    def forward(self, x):
        """
        x shape:
        [batch, 3, 224, 224]

        output:
        [batch, 512]
        """

        x = self.features(x)

        x = torch.flatten(
            x,
            start_dim=1
        )

        return x

In [16]:
class SpatialFrequencyFusion(nn.Module):

    def __init__(self):
        super().__init__()

        self.spatial_branch = (
            ResNet18FeatureExtractor()
        )

        self.frequency_branch = (
            ResNet18FeatureExtractor()
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                512 + 512,
                256
            ),

            nn.ReLU(),

            nn.Dropout(
                0.5
            ),

            nn.Linear(
                256,
                2
            )
        )

    def extract_video_features(
        self,
        frames,
        branch
    ):
        """
        frames:
        [B, 8, 3, 224, 224]

        Returns:
        [B, 512]
        """

        B, T, C, H, W = frames.shape

        # Process all frames together
        frames = frames.view(
            B * T,
            C,
            H,
            W
        )

        features = branch(frames)

        # [B*T, 512]
        features = features.view(
            B,
            T,
            512
        )

        # Mean temporal aggregation
        features = features.mean(
            dim=1
        )

        # [B, 512]
        return features

    def forward(
        self,
        rgb_frames,
        frequency_frames
    ):

        spatial_features = (
            self.extract_video_features(
                rgb_frames,
                self.spatial_branch
            )
        )

        frequency_features = (
            self.extract_video_features(
                frequency_frames,
                self.frequency_branch
            )
        )

        fused_features = torch.cat(
            [
                spatial_features,
                frequency_features
            ],
            dim=1
        )

        logits = self.classifier(
            fused_features
        )

        return logits

In [17]:
fusion_model = SpatialFrequencyFusion()

fusion_model = fusion_model.to(device)

print(fusion_model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 127MB/s]


SpatialFrequencyFusion(
  (spatial_branch): ResNet18FeatureExtractor(
    (features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  

In [18]:
fusion_model.eval()

with torch.no_grad():

    rgb_batch = rgb_batch.to(device)

    freq_batch = freq_batch.to(device)

    output = fusion_model(
        rgb_batch,
        freq_batch
    )

print(
    "Output shape:",
    output.shape
)

print(
    "Output:",
    output
)

Output shape: torch.Size([4, 2])
Output: tensor([[ 0.0026, -0.1116],
        [ 0.0643,  0.0743],
        [-0.3096, -0.0054],
        [ 0.2053, -0.0553]], device='cuda:0')


In [19]:
spatial_checkpoint = torch.load(
    SPATIAL_CHECKPOINT,
    map_location=device
)

print(spatial_checkpoint.keys())

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_f1'])


In [20]:
fusion_model.spatial_branch.features.load_state_dict(
    spatial_checkpoint["model_state_dict"],
    strict=False
)

print("Spatial pretrained weights loaded.")

Spatial pretrained weights loaded.


In [21]:
frequency_checkpoint = torch.load(
    FREQUENCY_CHECKPOINT,
    map_location=device
)

print(frequency_checkpoint.keys())

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_f1'])


In [22]:
fusion_model.frequency_branch.features.load_state_dict(
    frequency_checkpoint["model_state_dict"],
    strict=False
)

print("Frequency pretrained weights loaded.")

Frequency pretrained weights loaded.


In [23]:
class_counts = train_df["Label"].value_counts()

real_count = class_counts["REAL"]
fake_count = class_counts["FAKE"]

total = real_count + fake_count

fusion_class_weights = torch.tensor(
    [
        total / (2 * real_count),
        total / (2 * fake_count)
    ],
    dtype=torch.float32
).to(device)

print("REAL weight:", fusion_class_weights[0].item())
print("FAKE weight:", fusion_class_weights[1].item())

REAL weight: 2.9697985649108887
FAKE weight: 0.601222813129425


In [24]:
fusion_criterion = nn.CrossEntropyLoss(
    weight=fusion_class_weights
)

In [25]:
for param in fusion_model.spatial_branch.parameters():
    param.requires_grad = False

for param in fusion_model.frequency_branch.parameters():
    param.requires_grad = False

for param in fusion_model.classifier.parameters():
    param.requires_grad = True

In [26]:
trainable_params = sum(
    p.numel()
    for p in fusion_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in fusion_model.parameters()
)

print("Trainable parameters:", trainable_params)
print("Total parameters:", total_params)

Trainable parameters: 262914
Total parameters: 22615938


In [27]:
fusion_optimizer = torch.optim.AdamW(
    fusion_model.classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [28]:
def train_fusion_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    scaler=None
):

    model.train()

    running_loss = 0.0
    all_labels = []
    all_preds = []

    for rgb_frames, freq_frames, labels in loader:

        rgb_frames = rgb_frames.to(
            device,
            non_blocking=True
        )

        freq_frames = freq_frames.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        if scaler is not None:

            with torch.amp.autocast(
                device_type="cuda"
            ):

                logits = model(
                    rgb_frames,
                    freq_frames
                )

                loss = criterion(
                    logits,
                    labels
                )

            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            logits = model(
                rgb_frames,
                freq_frames
            )

            loss = criterion(
                logits,
                labels
            )

            loss.backward()

            optimizer.step()

        running_loss += (
            loss.item() *
            labels.size(0)
        )

        preds = torch.argmax(
            logits,
            dim=1
        )

        all_labels.extend(
            labels.detach()
            .cpu()
            .numpy()
        )

        all_preds.extend(
            preds.detach()
            .cpu()
            .numpy()
        )

    epoch_loss = (
        running_loss /
        len(loader.dataset)
    )

    epoch_acc = accuracy_score(
        all_labels,
        all_preds
    )

    return epoch_loss, epoch_acc

In [29]:
def evaluate_fusion(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():

        for rgb_frames, freq_frames, labels in loader:

            rgb_frames = rgb_frames.to(
                device,
                non_blocking=True
            )

            freq_frames = freq_frames.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            logits = model(
                rgb_frames,
                freq_frames
            )

            loss = criterion(
                logits,
                labels
            )

            running_loss += (
                loss.item() *
                labels.size(0)
            )

            probs = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_probs.extend(
                probs.cpu().numpy()
            )

    epoch_loss = (
        running_loss /
        len(loader.dataset)
    )

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    precision = precision_score(
        all_labels,
        all_preds,
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_preds,
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_preds,
        zero_division=0
    )

    auc = roc_auc_score(
        all_labels,
        all_probs
    )

    return {
        "loss": epoch_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "labels": np.array(all_labels),
        "preds": np.array(all_preds),
        "probs": np.array(all_probs)
    }

In [30]:
WARMUP_EPOCHS = 3

fusion_scaler = torch.amp.GradScaler("cuda")

for epoch in range(WARMUP_EPOCHS):

    train_loss, train_acc = train_fusion_one_epoch(
        fusion_model,
        fusion_train_loader,
        fusion_criterion,
        fusion_optimizer,
        device,
        fusion_scaler
    )

    val_metrics = evaluate_fusion(
        fusion_model,
        fusion_val_loader,
        fusion_criterion,
        device
    )

    print(
        f"Warmup Epoch [{epoch+1}/{WARMUP_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"Val AUC: {val_metrics['auc']:.4f}"
    )

Warmup Epoch [1/3] | Train Loss: 0.7642 | Train Acc: 0.6932 | Val Loss: 0.6667 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.5774
Warmup Epoch [2/3] | Train Loss: 0.6780 | Train Acc: 0.8316 | Val Loss: 0.6677 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.5851
Warmup Epoch [3/3] | Train Loss: 0.6636 | Train Acc: 0.8316 | Val Loss: 0.6527 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.5803


In [31]:
# Unfreeze both feature branches
for param in fusion_model.spatial_branch.parameters():
    param.requires_grad = True

for param in fusion_model.frequency_branch.parameters():
    param.requires_grad = True

# Keep fusion classifier trainable
for param in fusion_model.classifier.parameters():
    param.requires_grad = True

# Lower learning rate for pretrained branches
fusion_optimizer = torch.optim.AdamW(
    fusion_model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

print("Spatial branch trainable:", any(
    p.requires_grad for p in fusion_model.spatial_branch.parameters()
))

print("Frequency branch trainable:", any(
    p.requires_grad for p in fusion_model.frequency_branch.parameters()
))

print("Classifier trainable:", any(
    p.requires_grad for p in fusion_model.classifier.parameters()
))

Spatial branch trainable: True
Frequency branch trainable: True
Classifier trainable: True


In [33]:
from torch.utils.data import DataLoader

# Recreate paired datasets
train_dataset = SpatialFrequencyDataset(
    train_df,
    FACES_DIR,
    rgb_transform=rgb_transform,
    frequency_transform=frequency_transform
)

val_dataset = SpatialFrequencyDataset(
    val_df,
    FACES_DIR,
    rgb_transform=rgb_transform,
    frequency_transform=frequency_transform
)

test_dataset = SpatialFrequencyDataset(
    test_df,
    FACES_DIR,
    rgb_transform=rgb_transform,
    frequency_transform=frequency_transform
)

# Recreate DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 443
Validation batches: 98
Test batches: 105


In [34]:
FUSION_EPOCHS = 10

best_val_f1 = 0.0
best_fusion_state = None
fusion_history = []

fusion_scaler = torch.amp.GradScaler("cuda")

for epoch in range(FUSION_EPOCHS):

    train_loss, train_acc = train_fusion_one_epoch(
        fusion_model,
        train_loader,
        fusion_criterion,
        fusion_optimizer,
        device,
        fusion_scaler
    )

    val_metrics = evaluate_fusion(
        fusion_model,
        val_loader,
        fusion_criterion,
        device
    )

    print(
        f"Fusion Epoch [{epoch+1}/{FUSION_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"Val AUC: {val_metrics['auc']:.4f}"
    )

    fusion_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_f1": val_metrics["f1"],
        "val_auc": val_metrics["auc"]
    })

    # Save best model based on validation F1
    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_fusion_state = copy.deepcopy(fusion_model.state_dict())
        print("  ✓ Best fusion model updated")


Fusion Epoch [1/10] | Train Loss: 0.6520 | Train Acc: 0.8316 | Val Loss: 0.6497 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.6202
  ✓ Best fusion model updated
Fusion Epoch [2/10] | Train Loss: 0.6449 | Train Acc: 0.8316 | Val Loss: 0.6379 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.6702
Fusion Epoch [3/10] | Train Loss: 0.6363 | Train Acc: 0.8316 | Val Loss: 0.6267 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.6833
Fusion Epoch [4/10] | Train Loss: 0.6131 | Train Acc: 0.8316 | Val Loss: 0.6284 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.6502
Fusion Epoch [5/10] | Train Loss: 0.5954 | Train Acc: 0.8316 | Val Loss: 0.6028 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.7082
Fusion Epoch [6/10] | Train Loss: 0.5717 | Train Acc: 0.8316 | Val Loss: 0.5810 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.7462
Fusion Epoch [7/10] | Train Loss: 0.5512 | Train Acc: 0.8316 | Val Loss: 0.5654 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.7702
Fusion Epoch [8/10] | Train Loss: 0.529

In [35]:
# Inspect final fusion model on validation set
final_val_metrics = evaluate_fusion(
    fusion_model,
    val_loader,
    fusion_criterion,
    device
)

print("Validation Accuracy :", final_val_metrics["accuracy"])
print("Validation Precision:", final_val_metrics["precision"])
print("Validation Recall   :", final_val_metrics["recall"])
print("Validation F1       :", final_val_metrics["f1"])
print("Validation AUC      :", final_val_metrics["auc"])

print("\nValidation Confusion Matrix:")
print(confusion_matrix(
    final_val_metrics["labels"],
    final_val_metrics["preds"]
))

print("\nPredicted class counts:")
unique, counts = np.unique(
    final_val_metrics["preds"],
    return_counts=True
)

for cls, count in zip(unique, counts):
    print("REAL" if cls == 0 else "FAKE", ":", count)

Validation Accuracy : 0.8290816326530612
Validation Precision: 0.8482384823848238
Validation Recall   : 0.9660493827160493
Validation F1       : 0.9033189033189033
Validation AUC      : 0.7703794480755266

Validation Confusion Matrix:
[[ 12  56]
 [ 11 313]]

Predicted class counts:
REAL : 23
FAKE : 369


In [36]:
from sklearn.metrics import precision_recall_fscore_support

val_labels = final_val_metrics["labels"]
val_probs = final_val_metrics["probs"]

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):
    preds = (val_probs >= threshold).astype(int)

    precision = precision_score(
        val_labels, preds, zero_division=0
    )
    recall = recall_score(
        val_labels, preds, zero_division=0
    )
    f1 = f1_score(
        val_labels, preds, zero_division=0
    )
    accuracy = accuracy_score(
        val_labels, preds
    )

    threshold_results.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.to_string(index=False))

 threshold  accuracy  precision   recall       f1
      0.10  0.826531   0.826531 1.000000 0.905028
      0.15  0.826531   0.826531 1.000000 0.905028
      0.20  0.826531   0.826531 1.000000 0.905028
      0.25  0.826531   0.826531 1.000000 0.905028
      0.30  0.826531   0.826531 1.000000 0.905028
      0.35  0.826531   0.826531 1.000000 0.905028
      0.40  0.826531   0.826531 1.000000 0.905028
      0.45  0.829082   0.828645 1.000000 0.906294
      0.50  0.829082   0.848238 0.966049 0.903319
      0.55  0.765306   0.889262 0.817901 0.852090
      0.60  0.711735   0.892193 0.740741 0.809444
      0.65  0.663265   0.900000 0.666667 0.765957
      0.70  0.630102   0.916279 0.608025 0.730983
      0.75  0.576531   0.948864 0.515432 0.668000
      0.80  0.517857   0.965517 0.432099 0.597015
      0.85  0.446429   0.981982 0.336420 0.501149
      0.90  0.357143   1.000000 0.222222 0.363636


In [37]:
FUSION_CHECKPOINT = os.path.join(
    PROJECT_DIR,
    "fusion_resnet18_best_f1.pth"
)

torch.save({
    "model_state_dict": best_fusion_state,
    "val_f1": best_val_f1
}, FUSION_CHECKPOINT)

print("Saved:", FUSION_CHECKPOINT)
print("Best validation F1:", best_val_f1)

Saved: /content/drive/MyDrive/Multimedia_Authentication/fusion_resnet18_best_f1.pth
Best validation F1: 0.9050279329608939


In [38]:
fusion_model.load_state_dict(best_fusion_state)

test_metrics = evaluate_fusion(
    fusion_model,
    test_loader,
    fusion_criterion,
    device
)

print("TEST RESULTS")
print("-------------------------")
print("Accuracy :", test_metrics["accuracy"])
print("Precision:", test_metrics["precision"])
print("Recall   :", test_metrics["recall"])
print("F1       :", test_metrics["f1"])
print("ROC-AUC  :", test_metrics["auc"])

print("\nConfusion Matrix:")
print(confusion_matrix(
    test_metrics["labels"],
    test_metrics["preds"]
))

TEST RESULTS
-------------------------
Accuracy : 0.84688995215311
Precision: 0.84688995215311
Recall   : 1.0
F1       : 0.917098445595855
ROC-AUC  : 0.6287959039548023

Confusion Matrix:
[[  0  64]
 [  0 354]]


In [39]:
# Recreate a fresh fusion model
fusion_model = SpatialFrequencyFusion().to(device)

# Load pretrained spatial and frequency branches
spatial_checkpoint = torch.load(
    SPATIAL_CHECKPOINT,
    map_location=device
)

frequency_checkpoint = torch.load(
    FREQUENCY_CHECKPOINT,
    map_location=device
)

fusion_model.spatial_branch.features.load_state_dict(
    spatial_checkpoint["model_state_dict"],
    strict=False
)

fusion_model.frequency_branch.features.load_state_dict(
    frequency_checkpoint["model_state_dict"],
    strict=False
)

print("Fresh fusion model created.")
print("Pretrained spatial + frequency weights loaded.")

Fresh fusion model created.
Pretrained spatial + frequency weights loaded.


In [40]:
class_counts = train_df["Label"].value_counts()

real_count = class_counts["REAL"]
fake_count = class_counts["FAKE"]

total = real_count + fake_count

fusion_class_weights = torch.tensor(
    [
        total / (2 * real_count),
        total / (2 * fake_count)
    ],
    dtype=torch.float32
).to(device)

fusion_criterion = nn.CrossEntropyLoss(
    weight=fusion_class_weights
)

# Freeze pretrained branches during warm-up
for param in fusion_model.spatial_branch.parameters():
    param.requires_grad = False

for param in fusion_model.frequency_branch.parameters():
    param.requires_grad = False

for param in fusion_model.classifier.parameters():
    param.requires_grad = True

fusion_optimizer = torch.optim.AdamW(
    fusion_model.classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print("REAL weight:", fusion_class_weights[0].item())
print("FAKE weight:", fusion_class_weights[1].item())

REAL weight: 2.9697985649108887
FAKE weight: 0.601222813129425


In [41]:
WARMUP_EPOCHS = 3

fusion_scaler = torch.amp.GradScaler("cuda")

for epoch in range(WARMUP_EPOCHS):

    train_loss, train_acc = train_fusion_one_epoch(
        fusion_model,
        train_loader,
        fusion_criterion,
        fusion_optimizer,
        device,
        fusion_scaler
    )

    val_metrics = evaluate_fusion(
        fusion_model,
        val_loader,
        fusion_criterion,
        device
    )

    print(
        f"Warmup Epoch [{epoch+1}/{WARMUP_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"Val AUC: {val_metrics['auc']:.4f}"
    )

Warmup Epoch [1/3] | Train Loss: 0.8065 | Train Acc: 0.6949 | Val Loss: 0.6680 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.5684
Warmup Epoch [2/3] | Train Loss: 0.6635 | Train Acc: 0.8316 | Val Loss: 0.6510 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.5728
Warmup Epoch [3/3] | Train Loss: 0.6617 | Train Acc: 0.8316 | Val Loss: 0.6570 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.5945


In [42]:
# Unfreeze both branches
for param in fusion_model.spatial_branch.parameters():
    param.requires_grad = True

for param in fusion_model.frequency_branch.parameters():
    param.requires_grad = True

for param in fusion_model.classifier.parameters():
    param.requires_grad = True

# Small learning rate for pretrained networks
fusion_optimizer = torch.optim.AdamW(
    fusion_model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

print("Full fusion model unfrozen.")

Full fusion model unfrozen.


In [43]:
FUSION_EPOCHS = 10

best_val_auc = -1.0
best_fusion_auc_state = None
fusion_auc_history = []

fusion_scaler = torch.amp.GradScaler("cuda")

for epoch in range(FUSION_EPOCHS):

    train_loss, train_acc = train_fusion_one_epoch(
        fusion_model,
        train_loader,
        fusion_criterion,
        fusion_optimizer,
        device,
        fusion_scaler
    )

    val_metrics = evaluate_fusion(
        fusion_model,
        val_loader,
        fusion_criterion,
        device
    )

    print(
        f"Fusion-AUC Epoch [{epoch+1}/{FUSION_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"Val AUC: {val_metrics['auc']:.4f}"
    )

    fusion_auc_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_f1": val_metrics["f1"],
        "val_auc": val_metrics["auc"]
    })

    # Select checkpoint using validation ROC-AUC
    if val_metrics["auc"] > best_val_auc:
        best_val_auc = val_metrics["auc"]
        best_fusion_auc_state = copy.deepcopy(
            fusion_model.state_dict()
        )

        print("  ✓ Best AUC fusion model updated")

Fusion-AUC Epoch [1/10] | Train Loss: 0.6525 | Train Acc: 0.8316 | Val Loss: 0.6519 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.6000
  ✓ Best AUC fusion model updated
Fusion-AUC Epoch [2/10] | Train Loss: 0.6473 | Train Acc: 0.8316 | Val Loss: 0.6385 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.6781
  ✓ Best AUC fusion model updated
Fusion-AUC Epoch [3/10] | Train Loss: 0.6336 | Train Acc: 0.8322 | Val Loss: 0.6162 | Val Acc: 0.8265 | Val F1: 0.9050 | Val AUC: 0.6847
  ✓ Best AUC fusion model updated
Fusion-AUC Epoch [4/10] | Train Loss: 0.6024 | Train Acc: 0.8271 | Val Loss: 0.5900 | Val Acc: 0.8163 | Val F1: 0.8971 | Val AUC: 0.7119
  ✓ Best AUC fusion model updated
Fusion-AUC Epoch [5/10] | Train Loss: 0.5541 | Train Acc: 0.8345 | Val Loss: 0.5764 | Val Acc: 0.8112 | Val F1: 0.8896 | Val AUC: 0.7243
  ✓ Best AUC fusion model updated
Fusion-AUC Epoch [6/10] | Train Loss: 0.5086 | Train Acc: 0.8367 | Val Loss: 0.5571 | Val Acc: 0.8342 | Val F1: 0.9040 | Val AUC: 0.7685
  ✓ Best

In [44]:
# Restore the best validation-AUC checkpoint
fusion_model.load_state_dict(best_fusion_auc_state)

print("Best validation AUC:", best_val_auc)

# Evaluate once on the untouched test set
test_fusion_auc_metrics = evaluate_fusion(
    fusion_model,
    test_loader,
    fusion_criterion,
    device
)

print("\nTEST RESULTS — AUC-selected Fusion")
print("-----------------------------------")
print("Accuracy :", test_fusion_auc_metrics["accuracy"])
print("Precision:", test_fusion_auc_metrics["precision"])
print("Recall   :", test_fusion_auc_metrics["recall"])
print("F1       :", test_fusion_auc_metrics["f1"])
print("ROC-AUC  :", test_fusion_auc_metrics["auc"])

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        test_fusion_auc_metrics["labels"],
        test_fusion_auc_metrics["preds"]
    )
)

Best validation AUC: 0.8075980392156863

TEST RESULTS — AUC-selected Fusion
-----------------------------------
Accuracy : 0.8492822966507177
Precision: 0.896457765667575
Recall   : 0.9293785310734464
F1       : 0.912621359223301
ROC-AUC  : 0.784030720338983

Confusion Matrix:
[[ 26  38]
 [ 25 329]]
